In [ ]:
import os
import numpy as np


from PIL import Image, ImageDraw
import random


%matplotlib inline


In [124]:
def create_solvable_maze(width: int, height: int) -> np.ndarray:
    """
    Generates a solvable maze grid using a randomized depth-first search (DFS) algorithm.
    """
    maze = np.ones((height, width), dtype=np.uint8)

    # Starting point for DFS
    start_x, start_y = (1, 1)
    stack = [(start_x, start_y)]
    maze[start_y, start_x] = 0

    while stack:
        x, y = stack[-1]

        # Define possible directions (up, down, left, right)
        neighbors = []
        if x > 1 and maze[y, x - 2] == 1:
            neighbors.append((x - 2, y))
        if x < width - 2 and maze[y, x + 2] == 1:
            neighbors.append((x + 2, y))
        if y > 1 and maze[y - 2, x] == 1:
            neighbors.append((x, y - 2))
        if y < height - 2 and maze[y + 2, x] == 1:
            neighbors.append((x, y + 2))

        if neighbors:
            nx, ny = random.choice(neighbors)

            # Carve a path
            path_x = (x + nx) // 2
            path_y = (y + ny) // 2
            maze[path_y, path_x] = 0
            maze[ny, nx] = 0
            stack.append((nx, ny))
        else:
            stack.pop()

    return maze


def generate_maze_image(maze: np.ndarray, filename: str, doors: int = 1) -> None:
    """
    Converts a maze grid into a PNG image with black walls and blue doors.
    Compatible with extract_wall_mask and detect_doors.
    """
    height, width = maze.shape
    scale = 20  # Pixels per cell
    img_width, img_height = width * scale, height * scale

    # White background
    image = Image.new("RGB", (img_width, img_height), color=(255, 255, 255))
    draw = ImageDraw.Draw(image)

    # Black walls
    for y in range(height):
        for x in range(width):
            if maze[y, x] == 1:
                draw.rectangle(
                    [x * scale, y * scale, (x + 1) * scale, (y + 1) * scale],
                    fill=(0, 0, 0),  # black
                )

    # Door placement
    door_locations = []
    possible_doors = []

    # Top/bottom walls
    for x in range(1, width - 1, 2):
        if maze[1, x] == 0:
            possible_doors.append((x, 0))  # Top
        if maze[height - 2, x] == 0:
            possible_doors.append((x, height - 1))  # Bottom

    # Left/right walls
    for y in range(1, height - 1, 2):
        if maze[y, 1] == 0:
            possible_doors.append((0, y))  # Left
        if maze[y, width - 2] == 0:
            possible_doors.append((width - 1, y))  # Right

    num_doors = min(doors, len(possible_doors))
    selected_doors = random.sample(possible_doors, num_doors)

    # Door color compatible with detect_doors (BGR-ish blue)
    door_color = (247, 0, 0)  # Will match your detect_doors range [0,0,245]–[0,19,249]

    for x, y in selected_doors:
        door_x_start = x * scale
        door_y_start = y * scale
        draw.rectangle(
            [door_x_start, door_y_start, door_x_start + scale, door_y_start + scale],
            fill=door_color,
        )
        # Center coordinates
        door_locations.append((x * scale + scale // 2, y * scale + scale // 2))

    image.save(filename)
    print(f"Maze saved as {filename} with {len(door_locations)} doors.")


In [129]:
if not os.path.exists("mazes"):
    os.makedirs("mazes")

# Generate a few mazes with varying sizes and door counts
generate_maze_image(create_solvable_maze(31, 31), "mazes/maze_31x31_1door.png", doors=1)
generate_maze_image(
    create_solvable_maze(41, 41), "mazes/maze_41x41_2doors.png", doors=2
)
generate_maze_image(
    create_solvable_maze(51, 51), "mazes/maze_51x51_3doors.png", doors=3
)
generate_maze_image(
    create_solvable_maze(51, 51), "mazes/maze_51x51_4doors.png", doors=4
)
generate_maze_image(
    create_solvable_maze(51, 51), "mazes/maze_51x51_1doors.png", doors=1
)
generate_maze_image(
    create_solvable_maze(51, 51), "mazes/maze_51x51_2doors.png", doors=2
)


Maze saved as mazes/maze_31x31_1door.png with 1 doors.
Maze saved as mazes/maze_41x41_2doors.png with 2 doors.
Maze saved as mazes/maze_51x51_3doors.png with 3 doors.
Maze saved as mazes/maze_51x51_4doors.png with 4 doors.
Maze saved as mazes/maze_51x51_1doors.png with 1 doors.
Maze saved as mazes/maze_51x51_2doors.png with 2 doors.
